# OdontoCA YOLO Automation Notebook

This notebook provides a **fully automated**, stepâ€‘byâ€‘step pipeline for training a YOLOv8 model on the OdontoCA dental image dataset, evaluating performance, and packaging the best model for distribution.

It is designed to be run in **Google Colab** with a mounted Google Drive containing the `OdontoCA` data folder. Each section includes explanatory markdown cells to aid understanding (didactic).

---


In [ ]:
# 1ï¸âƒ£ Setup â€“ mount Drive and install dependencies
from pathlib import Path
import os, json, hashlib, zipfile

# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive already mounted or not in Colab:', e)

# Define project root (adjust if needed)
PROJECT_ROOT = Path('/content/drive/MyDrive/OdontoCA')
DATA_DIR = PROJECT_ROOT / 'data'  # adjust if your data lives elsewhere

# Install required packages
!pip install -q ultralytics tqdm pandas matplotlib seaborn

# Verify installation
import ultralytics
print('ultralytics version:', ultralytics.__version__)


---
## 2ï¸âƒ£ Configuration â€“ hyperâ€‘parameters, paths, and class weighting

We define the training configuration in a dictionary that will be passed to the YOLO API. Feel free to edit any values to experiment with different settings.


In [ ]:
from ultralytics import YOLO
import yaml, pandas as pd

# Paths â€“ adjust if your folder structure differs
train_path = str(DATA_DIR / 'images' / 'train')
val_path   = str(DATA_DIR / 'images' / 'val')
yaml_path  = str(PROJECT_ROOT / 'data' / 'odontoca.yaml')  # dataset yaml for YOLO

# Create YOLO dataset yaml if it does not exist
if not Path(yaml_path).exists():
    dataset_cfg = {
        'train': train_path,
        'val': val_path,
        'nc': 2,  # number of classes (e.g., caries, healthy)
        'names': ['caries', 'healthy']
    }
    with open(yaml_path, 'w') as f:
        yaml.safe_dump(dataset_cfg, f)
    print('Created dataset yaml at', yaml_path)
else:
    print('Dataset yaml already exists')

# Hyperâ€‘parameters â€“ feel free to tune
hyp = {
    'lr0': 0.01,          # initial learning rate
    'momentum': 0.937,   # SGD momentum/Adam beta1
    'weight_decay': 0.0005,
    'obj': 0.7,          # object loss weight (increase to focus on detection)
    'cls': 0.3,          # class loss weight (decrease for class imbalance)
    'iou': 0.20,         # IoU loss gain
    'anchor_t': 4.0,     # anchor threshold
    'scale': 0.5,        # image scale factor (1024px)
    'epochs': 100,
    'batch': 16,
    'imgsz': 1024,
    'patience': 10,      # earlyâ€‘stopping patience
    'proj': 'OdontoCA_yolo',
    'name': 'exp',
    'save': True,
    'verbose': True
}

# Load preâ€‘trained YOLOv8s model
model = YOLO('yolov8s.pt')


---
## 3ï¸âƒ£ Training â€“ oneâ€‘line call with callbacks

The training will automatically log metrics to `runs/train` and stop early if the validation loss does not improve for `patience` epochs.


In [ ]:
# Train the model
results = model.train(
    data=yaml_path,
    epochs=hyp['epochs'],
    batch=hyp['batch'],
    imgsz=hyp['imgsz'],
    lr0=hyp['lr0'],
    momentum=hyp['momentum'],
    weight_decay=hyp['weight_decay'],
    obj=hyp['obj'],
    cls=hyp['cls'],
    iou=hyp['iou'],
    patience=hyp['patience'],
    project=PROJECT_ROOT / 'runs' / 'train',
    name='yolo_automation',
    exist_ok=True
)

# The best model is saved as `best.pt` in the run folder
best_path = results.best
print('Best model saved to', best_path)

---
## 4ï¸âƒ£ Evaluation â€“ compute mAP@0.5 and generate results CSV

We use the builtâ€‘in YOLO validation to obtain the metrics and then export a concise CSV for reporting.


In [ ]:
# Validation (uses the same validation split)
metrics = model.val(data=yaml_path, imgsz=hyp['imgsz'], batch=hyp['batch'], name='val_automation')

# Extract mAP@0.5
map50 = metrics.box.map50 * 100
print(f'ðŸª„ mAP@0.5 = {map50:.2f}%')

# Save results to CSV
results_df = pd.DataFrame({
    'Metric': ['mAP@0.5'],
    'Value': [map50]
})
csv_path = PROJECT_ROOT / 'results.csv'
results_df.to_csv(csv_path, index=False)
print('Results saved to', csv_path)

---
## 5ï¸âƒ£ Postâ€‘processing â€“ hashing, zipping, and optional GitHub push

* Compute a SHAâ€‘256 hash of the best model for reproducibility.
* Package the model and its `data.yaml` into a ZIP file.
* (Optional) Trigger the existing `push_repo.ps1` PowerShell script to create a GitHub release.


In [ ]:
# Compute SHAâ€‘256 hash of best.pt
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

model_hash = sha256_file(best_path)
print('SHAâ€‘256:', model_hash)

# Create ZIP containing best.pt and dataset yaml
zip_path = PROJECT_ROOT / f'OdontoCA_yolo_{model_hash[:8]}.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    zipf.write(best_path, arcname='best.pt')
    zipf.write(yaml_path, arcname='odontoca.yaml')
print('Package created at', zip_path)

# OPTIONAL: push to GitHub â€“ uncomment the block below if you want an automatic push
# import subprocess
# subprocess.run(['powershell', '-File', str(PROJECT_ROOT / 'push_repo.ps1'), str(zip_path)], check=True)

---
## 6ï¸âƒ£ Didactic Commentary â€“ tips and next steps

* **Hyperâ€‘parameter tuning** â€“ increase `epochs` or adjust `lr0` if the mAP is below the target.
* **Data augmentation** â€“ add `augment=True` in the `model.train` call to improve robustness.
* **Class weighting** â€“ the `obj` and `cls` values above help mitigate class imbalance typical in dental caries datasets.
* **Model export** â€“ after training you can export to ONNX/TorchScript with `model.export(format='onnx')` for deployment on edge devices.

Feel free to explore the **runs/train** folder for TensorBoard logs, sample predictions, and more visual diagnostics.